# 05 · Embeddings, RNNs and the Road to Attention

Follow the path that led to transformers: turn words into vectors (`nn.Embedding`, then word embeddings trained from
scratch), read sequences with an RNN/LSTM, measure the two things that break (the fixed-size bottleneck and gradients
vanishing through time), and fix them with attention, whose weights you can plot and read.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · training loop](02_training_loop_optimizers_lr_schedules.ipynb), [01 · backprop and vanishing gradients](01_neuron_and_backprop_from_scratch.ipynb)

## Why this matters in interviews

- "What is an embedding?" and "how does a token id become a vector?" open most LLM interviews. The precise answer
  is a lookup table that equals a one-hot vector times a matrix, and you can prove it in one line.
- "Why did transformers replace RNNs?" needs two reasons with evidence: parallelism, and a direct path between
  distant tokens (no bottleneck, no vanishing gradient through time).
- Attention explained as the fix for a concrete failure is the clearest way to teach it, which is exactly what an
  interviewer tests when they ask you to explain attention.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo04` | What is an embedding? |
| `tr37` | How does a token id become a vector before self-attention runs? |
| `tr25` | What is the difference between a transformer and an RNN/LSTM, and why did transformers win? |
| `tr03` | What is self-attention, in your own words? |
| `tr20` | What is cross-attention and where do you see it? |
| `ve01` | What is cosine similarity, and what are the alternatives? |

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(0); np.random.seed(0); torch.manual_seed(0)
torch.set_num_threads(1)                         # tiny models: single-threaded is fastest
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.8})
print("torch", torch.__version__)

## 1. `nn.Embedding` is a lookup table

**In plain English:** the model keeps one row of numbers per vocabulary entry. Looking up token id 3 hands you row
3. Those rows are ordinary parameters, learned by backprop like any weight.

Mathematically the lookup is a one-hot vector times the embedding matrix; in code it is a fast row gather.

In [ ]:
vocab_size, d_model = 10, 4
emb = nn.Embedding(vocab_size, d_model)
ids = torch.tensor([3, 7, 3])                              # a "sentence" of token ids

looked_up = emb(ids)                                       # [3, 4]: row 3, row 7, row 3
one_hot = F.one_hot(ids, vocab_size).float()               # [3, 10]
as_matmul = one_hot @ emb.weight                           # [3, 10] @ [10, 4] -> [3, 4]
print("emb(ids):\n", looked_up.detach().numpy().round(3))
assert torch.allclose(looked_up, as_matmul) and torch.equal(looked_up[0], emb.weight[3])

looked_up.sum().backward()
rows_with_grad = torch.nonzero(emb.weight.grad.abs().sum(1)).flatten().tolist()
print("rows that received a gradient:", rows_with_grad, "| row 3 grad (used twice):", emb.weight.grad[3].tolist())
assert rows_with_grad == [3, 7] and torch.all(emb.weight.grad[3] == 2)

Only the rows that were looked up get a gradient (row 3 twice, because it appeared twice): embedding updates are
sparse. In an LLM this table is `[vocab, d_model]` (for example 128,256 × 3,072 in Llama 3.2 3B), its output is
`[batch, seq, d_model]`, and position information is added separately (learned or sinusoidal embeddings at the
input, or RoPE inside attention), because the lookup alone has no idea of word order.

## 2. Word embeddings from scratch: skip-gram

**In plain English:** "you shall know a word by the company it keeps". Train each word's vector to predict the words
around it. Words used in similar contexts (dog, puppy) end up with similar vectors, without anyone telling the model
what the words mean. This is word2vec's skip-gram; modern embedding models use the same idea at vastly larger scale
with transformer encoders and contrastive objectives.

A small generated corpus about four topics:

In [ ]:
groups = {"animals": ["cat", "dog", "puppy", "kitten", "horse"],
          "royalty": ["king", "queen", "prince", "princess", "throne"],
          "fruit": ["apple", "banana", "mango", "orange", "grape"],
          "vehicles": ["car", "bus", "train", "truck", "bike"]}
templates = {"animals": ["the {} chased the ball", "my {} sleeps on the rug", "a hungry {} wants food", "the {} barked and ran"],
             "royalty": ["the {} ruled the kingdom", "a {} wore the crown", "the {} lives in the palace", "the {} gave a royal order"],
             "fruit": ["she ate a ripe {}", "a sweet {} from the market", "he sliced the fresh {}", "the {} juice was sweet"],
             "vehicles": ["the {} drove down the road", "we took the {} to work", "the {} needs fuel", "a fast {} on the highway"]}
rng = random.Random(0)
corpus = []
for _ in range(300):
    topic = rng.choice(list(groups))
    corpus.append(rng.choice(templates[topic]).format(rng.choice(groups[topic])).split())
vocab = sorted({w for sentence in corpus for w in sentence})
word_id = {w: i for i, w in enumerate(vocab)}

WINDOW = 2                                                 # context = up to 2 words each side
pairs = torch.tensor([(word_id[s[i]], word_id[s[j]]) for s in corpus for i in range(len(s))
                      for j in range(max(0, i - WINDOW), min(len(s), i + WINDOW + 1)) if j != i])
print(f"{len(corpus)} sentences, vocabulary {len(vocab)}, {len(pairs):,} (centre, context) pairs")
print("examples:", [" ".join(s) for s in corpus[:3]])
print("first pairs:", [(vocab[a], vocab[b]) for a, b in pairs[:4].tolist()])

In [ ]:
torch.manual_seed(0)
center_emb = nn.Embedding(len(vocab), 16)                 # the word vectors we want
context_head = nn.Linear(16, len(vocab))                  # predicts the context word from the centre vector
opt = torch.optim.Adam([*center_emb.parameters(), *context_head.parameters()], lr=0.05)
losses = []
for step in range(150):                                   # full batch: every pair, every step
    opt.zero_grad()
    loss = F.cross_entropy(context_head(center_emb(pairs[:, 0])), pairs[:, 1])
    loss.backward()
    opt.step()
    losses.append(loss.item())
print(f"skip-gram loss {losses[0]:.2f} -> {losses[-1]:.2f}  (can't reach 0: many contexts are equally likely)")

E = F.normalize(center_emb.weight.detach(), dim=1)        # unit vectors, so dot product = cosine similarity
words = [w for ws in groups.values() for w in ws]
topic_of = {w: t for t, ws in groups.items() for w in ws}
S = E[[word_id[w] for w in words]] @ E[[word_id[w] for w in words]].T
same = torch.tensor([[topic_of[a] == topic_of[b] for b in words] for a in words])
off_diag = ~torch.eye(len(words), dtype=torch.bool)
intra, inter = S[same & off_diag].mean().item(), S[~same].mean().item()
print(f"mean cosine similarity: same topic {intra:.2f}   different topics {inter:.2f}")
for w in ["dog", "queen", "mango", "bus"]:
    top = [vocab[i] for i in (E @ E[word_id[w]]).argsort(descending=True)[1:4]]
    print(f"nearest to {w:5s}: {top}")
    assert all(topic_of.get(t) == topic_of[w] for t in top)
assert intra > inter + 0.3

In [ ]:
X = E[[word_id[w] for w in words]].numpy()
Xc = X - X.mean(0)
_, _, Vt = np.linalg.svd(Xc, full_matrices=False)          # PCA via SVD
xy = Xc @ Vt[:2].T                                         # project 16-D vectors onto the top 2 directions
fig, ax = plt.subplots(figsize=(9, 5))
for k, (topic, ws) in enumerate(groups.items()):
    pts = np.array([xy[words.index(w)] for w in ws])
    ax.scatter(pts[:, 0], pts[:, 1], s=60, color=COLORS[k], label=topic, edgecolor="white", linewidth=1.5)
    offsets = [(6, 6), (6, -12), (-40, 6), (-40, -12), (6, 18)]      # stagger labels inside tight clusters
    for w, (a, b), off in zip(ws, pts, offsets):
        ax.annotate(w, (a, b), xytext=off, textcoords="offset points", fontsize=9)
ax.set(title="Skip-gram word vectors (16-D), projected to 2-D with PCA", xlabel="principal component 1",
       ylabel="principal component 2")
ax.legend(); plt.show()

Four clean clusters, learned only from which words appear near which. "Close in vector space" now means "used in
similar contexts", and cosine similarity (the dot product of unit vectors, ignoring length) is the standard way to
measure it; alternatives are the raw dot product (when magnitude carries meaning) and Euclidean distance (equivalent
ranking to cosine once vectors are normalised). Caveat: these vectors capture co-occurrence, not truth; "king" and
"throne" are neighbours because they share contexts, not because a throne is royalty.

## 3. Reading sequences: the RNN

**In plain English:** an RNN reads one token at a time and keeps a running summary, the hidden state `h`. Each step
mixes the new token into the old summary: `h_t = tanh(W_x x_t + W_h h_{t−1} + b)`. The same weights are reused at
every step. Written by hand, it matches `nn.RNN`:

In [ ]:
torch.manual_seed(0)
rnn = nn.RNN(input_size=3, hidden_size=5, batch_first=True)
x = torch.randn(2, 6, 3)                                   # [batch, time, features]
out_torch, h_last = rnn(x)

h = torch.zeros(2, 5)
states = []
for t in range(x.shape[1]):                                # strictly one step after another
    h = torch.tanh(x[:, t] @ rnn.weight_ih_l0.T + rnn.bias_ih_l0 + h @ rnn.weight_hh_l0.T + rnn.bias_hh_l0)
    states.append(h)
assert torch.allclose(torch.stack(states, 1), out_torch, atol=1e-6) and torch.allclose(h, h_last[0], atol=1e-6)
print("hand-written recurrence == nn.RNN; output", tuple(out_torch.shape), "final state", tuple(h_last.shape))

Two consequences are visible in that loop. Step `t` cannot start before step `t−1` finishes, so training cannot be
parallelised across the sequence. And everything the model knows about the past must fit in `h`, a fixed-size
vector (5 numbers here), however long the input.

An **LSTM** adds a separate *cell state* `c` updated additively, `c_t = f_t ⊙ c_{t−1} + i_t ⊙ g_t`, with learned
gates: forget `f`, input `i`, output `o` (each a sigmoid, between 0 and 1). The forget gate decides how much of the
old memory survives each step. GRUs are a lighter variant with two gates.

## 4. A toy task that exposes the bottleneck: reverse a sequence

Input: 12 random digits. Target: the same digits reversed. Two models, same LSTM encoder, same training budget:

- **Bottleneck:** read the whole sequence, then predict all 12 outputs from the *final hidden state only*
  (the classic encoder-decoder before 2014).
- **Attention:** each output position has a learned query that scores every encoder state, takes a softmax, and
  reads a weighted average of the states. The output can look straight back at any input position.

In [ ]:
L, V, H = 12, 10, 32                                       # sequence length, digit vocabulary, hidden size

class BottleneckLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb, self.lstm = nn.Embedding(V, 16), nn.LSTM(16, H, batch_first=True)
        self.out = nn.Linear(H, L * V)

    def forward(self, x):
        states, _ = self.lstm(self.emb(x))                 # [B, L, H]
        return self.out(states[:, -1]).view(-1, L, V), None   # everything must pass through the last state

class AttentionLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb, self.pos = nn.Embedding(V, 16), nn.Embedding(L, 16)   # position info, as transformers add it
        self.lstm = nn.LSTM(16, H, batch_first=True)
        self.queries = nn.Parameter(0.1 * torch.randn(L, H))           # one learned query per output position
        self.key = nn.Linear(H, H)
        self.out = nn.Linear(H, V)

    def forward(self, x):
        states, _ = self.lstm(self.emb(x) + self.pos.weight)            # [B, L, H]: values
        scores = self.queries @ self.key(states).transpose(1, 2) / H ** 0.5   # [B, L_out, L_in]
        weights = scores.softmax(dim=-1)                                # each row sums to 1
        context = weights @ states                                      # [B, L_out, H]: weighted average
        return self.out(context), weights

def train_reverser(model, steps=150, batch=128, lr=2e-2):
    opt, gen, losses = torch.optim.Adam(model.parameters(), lr=lr), torch.Generator().manual_seed(0), []
    for step in range(steps):
        x = torch.randint(0, V, (batch, L), generator=gen)
        logits, _ = model(x)
        loss = F.cross_entropy(logits.reshape(-1, V), x.flip(1).reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

models, curves = {}, {}
for name, cls in [("bottleneck (last state only)", BottleneckLSTM), ("attention over all states", AttentionLSTM)]:
    torch.manual_seed(0)
    models[name] = cls()
    t0 = time.time()
    curves[name] = train_reverser(models[name])
    print(f"{name:30s} trained 150 steps in {time.time() - t0:.1f}s, final loss {np.mean(curves[name][-10:]):.3f}")

In [ ]:
x_test = torch.randint(0, V, (1000, L), generator=torch.Generator().manual_seed(99))
per_pos = {}
for name, m in models.items():
    with torch.no_grad():
        logits, _ = m(x_test)
    per_pos[name] = (logits.argmax(-1) == x_test.flip(1)).float().mean(0).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for name in models:
    ax1.plot(curves[name], label=name)
    # output position t needs input position L-1-t, which was read t steps before the last input
    ax2.plot(range(L), per_pos[name], marker="o", markersize=4, label=name)
ax1.axhline(np.log(V), color="#898781", lw=1)
ax1.set(title="Training loss (grey: ln 10, random guessing)", xlabel="step", ylabel="cross-entropy")
ax2.set(title="Test accuracy by how long ago the needed digit was read", xlabel="steps before the end (0 = the last digit read)",
        ylabel="accuracy", ylim=(0, 1.05))
ax1.legend(); plt.tight_layout(); plt.show()
for name, acc in per_pos.items():
    print(f"{name:30s} mean accuracy {acc.mean():.3f}   per position {[round(float(a), 2) for a in acc]}")
bn, at = per_pos["bottleneck (last state only)"], per_pos["attention over all states"]
assert at.mean() > 0.97 and bn.mean() < 0.7 and bn[0] > 0.9 > bn[4:8].max()

The bottleneck model gets the most recent digit almost always right, then collapses to near guessing for digits read
a few steps earlier: squeezing 12 digits through one 32-number vector *and* learning to unpack it is hard, and more
training helps only slowly (a small bump at the far end is the first digit, remembered slightly better). With
attention, every output reads the state it needs directly, and the task is solved in the same 150 steps.

## 5. Vanishing gradients through time

Why is old information hard to learn? The gradient from the loss back to the input at step `t` passes through every
later step, multiplying by one Jacobian per step, the same product-of-factors problem as depth in
[01](01_neuron_and_backprop_from_scratch.ipynb). Measure `‖∂(final state)/∂x_t‖` for each step of a 40-step sequence,
at initialisation:

In [ ]:
def grad_through_time(model, T=40):
    x = torch.randn(64, T, 16, requires_grad=True)
    states, _ = model(x)
    states[:, -1].pow(2).sum().backward()                  # a loss that depends on the final state only
    g = x.grad.norm(dim=(0, 2))
    return (g / g[-1]).numpy()                             # relative to the most recent step

torch.manual_seed(0); vanilla = nn.RNN(16, 32, batch_first=True)
torch.manual_seed(0); lstm_default = nn.LSTM(16, 32, batch_first=True)
torch.manual_seed(0); lstm_open = nn.LSTM(16, 32, batch_first=True)
with torch.no_grad():
    lstm_open.bias_ih_l0[32:64].fill_(3.0)                 # PyTorch gate order i, f, g, o: this is the forget gate

flows = {"vanilla RNN (tanh)": grad_through_time(vanilla),
         "LSTM, default init (forget gate ~ 0.5)": grad_through_time(lstm_default),
         "LSTM, forget-gate bias +3 (gate ~ 0.95)": grad_through_time(lstm_open)}
fig, ax = plt.subplots(figsize=(10, 4))
for name, g in flows.items():
    ax.semilogy(range(1, 41), g, marker="o", markersize=3, label=name)
ax.set(title="How much the input at each step affects the final state", xlabel="time step (40 = most recent)",
       ylabel="relative gradient norm (log)")
ax.legend(); plt.show()
for name, g in flows.items():
    print(f"{name:42s} step-1 gradient relative to step-40: {g[0]:.1e}")
assert flows["vanilla RNN (tanh)"][0] < 1e-6 and flows["LSTM, forget-gate bias +3 (gate ~ 0.95)"][0] > 0.05

In a vanilla RNN the influence of step 1 on the final state is billions of times smaller than that of step 40, so
gradient descent barely learns anything about it. An LSTM at default initialisation is not automatically better: its
forget gate starts near σ(0) = 0.5, which halves the memory every step. The LSTM's real fix is that the cell-state
path is **additive and gated**: when the forget gate is near 1, gradients flow back almost unchanged (the same idea
as a residual connection). The model can *learn* to open that gate; initialising the forget bias to +1 is a common
trick for exactly this reason. Still, the path from step 1 to step 40 is 40 multiplications long, and the model has
to learn to keep the gate open.

## 6. Attention as the fix, and reading its weights

Attention scores every (query, key) pair, softmaxes the scores into weights that sum to 1, and returns the weighted
average of the values. In our reverser the queries are learned per output position, the keys and values come from
the LSTM states. Plot the average weights over the test set:

In [ ]:
with torch.no_grad():
    _, weights = models["attention over all states"](x_test)
avg = weights.mean(0).numpy()                               # [L_out, L_in]
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(avg, cmap="Blues", vmin=0, vmax=1)
ax.set(title="Attention weights: output position vs input position", xlabel="input position (key)",
       ylabel="output position (query)", xticks=range(L), yticks=range(L))
ax.grid(False); fig.colorbar(im, ax=ax, label="mean attention weight"); plt.show()

assert torch.allclose(weights.sum(-1), torch.ones(1000, L), atol=1e-5)          # softmax rows sum to 1
assert (avg.argmax(1) == np.arange(L)[::-1]).all()                             # output t attends to input L-1-t
print("each output's most-attended input:", avg.argmax(1).tolist())
print(f"mean weight on the correct input: {np.mean([avg[t, L - 1 - t] for t in range(L)]):.2f}")

A clean anti-diagonal: output 0 attends to input 11, output 11 to input 0. Nobody told the model this; it learned
where to look. That is the whole idea, and it scales:

- **Additive (Bahdanau, 2014)** attention scores with a small MLP, `v·tanh(W_q q + W_k k)`; **dot-product (Luong)**
  uses `q·k`; transformers use **scaled dot-product**, `softmax(QKᵀ/√d)V`, which is one matrix multiply.
- Here the queries come from the decoder side and the keys/values from the encoder states: that is
  **cross-attention**, the setup of the original seq2seq translators and of encoder-decoder transformers (T5,
  Whisper). **Self-attention** is the same computation where queries, keys and values all come from the same
  sequence, so every token can gather information from every other token.
- The transformer's step: drop the RNN entirely and let stacked self-attention do all the mixing, adding position
  information explicitly (as our `pos` embedding hinted). Build it in
  [Attention from scratch](../07_genai_foundations/03_attention_from_scratch.ipynb).

## 7. Why transformers won

| | RNN | LSTM / GRU | Transformer (self-attention) |
|---|---|---|---|
| sequential steps per layer (training) | n, one after another | n | 1: all positions in parallel |
| path length between distant tokens | n | n (gated, so survives longer) | 1 |
| memory of the past | one fixed-size vector | fixed-size vector + cell | every previous token's keys/values |
| compute per layer | O(n·d²) | O(n·d²) | O(n²·d) attention + O(n·d²) MLP |
| inference state | O(1), constant | O(1) | O(n) KV cache, grows with context |
| long-range dependencies | poor (vanishing gradients) | better, still limited | direct |
| where you see it now | teaching | small on-device / streaming models | every LLM |

The two reasons to say: **parallel training** on GPUs over the whole sequence, and **a constant-length path**
between any two tokens (no bottleneck, no gradient decay through time). The price is quadratic attention and a KV
cache that grows with context, which is why state-space models such as Mamba try to recover RNN-style linear cost.

## Try it yourself

**1.** Implement **additive (Bahdanau)** attention scores, `v · tanh(W_q q + W_k k)`, for the trained reverser's
queries and LSTM states. Check the shape `[batch, L_out, L_in]` and that the softmax rows sum to 1.

In [ ]:
# Solution — try it yourself first, then run this
m = models["attention over all states"]
with torch.no_grad():
    states, _ = m.lstm(m.emb(x_test[:4]) + m.pos.weight)            # [4, L, H]
torch.manual_seed(0)
W_q, W_k, v = nn.Linear(H, H, bias=False), nn.Linear(H, H, bias=False), nn.Linear(H, 1, bias=False)
q = W_q(m.queries)[None, :, None, :]                                 # [1, L_out, 1, H]
k = W_k(states)[:, None, :, :]                                       # [B, 1, L_in, H]
additive_scores = v(torch.tanh(q + k)).squeeze(-1)                  # broadcast to [B, L_out, L_in]
add_weights = additive_scores.softmax(-1)
print("additive attention weights:", tuple(add_weights.shape))
assert add_weights.shape == (4, L, L) and torch.allclose(add_weights.sum(-1), torch.ones(4, L), atol=1e-6)
print(f"largest weight {add_weights.max():.2f} (uniform would be {1 / L:.2f}): untrained W_q, W_k, v barely prefer any input;")
print("in a real model they are trained with everything else, exactly like the dot-product version above.")
assert add_weights.max() < 0.5

**2.** Write `most_similar(word, k)` over the skip-gram vectors using cosine similarity (without relying on `E` being
pre-normalised), and use it to list the 4 nearest words to "truck".

In [ ]:
# Solution — try it yourself first, then run this
def most_similar(word, k=4):
    W = center_emb.weight.detach()
    v = W[word_id[word]]
    cos = (W @ v) / (W.norm(dim=1) * v.norm())                      # cosine = dot / (|a| |b|)
    order = cos.argsort(descending=True)
    return [(vocab[i], round(cos[i].item(), 3)) for i in order[1:k + 1]]

neighbours = most_similar("truck")
print("nearest to truck:", neighbours)
assert sum(topic_of.get(w) == "vehicles" for w, _ in neighbours) >= 3

**3.** Why does the attention model need the position embedding `pos`? Predict, then train a copy without it (the
LSTM states must then encode position on their own) and compare accuracy after the same 150 steps.

In [ ]:
# Solution — try it yourself first, then run this
class AttentionNoPos(AttentionLSTM):
    def forward(self, x):
        states, _ = self.lstm(self.emb(x))                          # no position embedding
        weights = (self.queries @ self.key(states).transpose(1, 2) / H ** 0.5).softmax(-1)
        return self.out(weights @ states), weights

torch.manual_seed(0)
no_pos = AttentionNoPos()
train_reverser(no_pos)
with torch.no_grad():
    acc_no_pos = (no_pos(x_test)[0].argmax(-1) == x_test.flip(1)).float().mean().item()
print(f"with position embeddings {at.mean():.3f}   without {acc_no_pos:.3f}")
print("Attention by itself is order-blind: to find 'input 11' the keys must carry position. Here the LSTM recovers")
print("position from its own recurrence almost perfectly, so the gap is small; a transformer has no recurrence,")
print("so without position information it could not solve this task at all.")
assert acc_no_pos < at.mean()

## Say it in an interview

- **Embedding:** a learned `[vocab, d_model]` table; lookup = one-hot × matrix, implemented as a gather; only looked-up
  rows get gradients. Output `[batch, seq, d_model]`; position is added separately (or applied inside attention
  with RoPE).
- **Why similar words get similar vectors:** training pushes words with similar contexts to similar positions
  (skip-gram predicts context words; modern embedders use contrastive objectives). Compare vectors with cosine
  similarity.
- **RNN:** `h_t = tanh(W_x x_t + W_h h_{t−1} + b)`, strictly sequential. Two failures: a fixed-size bottleneck and
  gradients that vanish through time (a product of per-step Jacobians). LSTM gates make the memory path additive,
  which helps if the forget gate learns to stay open.
- **Attention in two sentences:** each position asks a question (query), every position advertises what it has
  (key) and what it will hand over (value); the output is the softmax-weighted average of values by query-key
  similarity. It gives every output a direct, learned path to every input.
- **Why transformers won:** parallel training across the sequence and a constant path length between tokens; the
  cost is O(n²) attention and a KV cache that grows with context.
- **Traps:** "attention understands order" (it does not, without position information: exercise 3); "LSTMs solve
  vanishing gradients" (they mitigate it, and only once the gates learn to keep memory); confusing cross-attention
  (queries from one sequence, keys/values from another) with self-attention.

## Next

- [Attention from scratch](../07_genai_foundations/03_attention_from_scratch.ipynb): scaled dot-product
  self-attention, masks and multiple heads.
- [Tokenization, BPE and tiktoken](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb) (where the token
  ids come from) · [Embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb) (modern
  sentence embeddings) · [Transformer block and tiny GPT](../07_genai_foundations/04_transformer_block_and_tiny_gpt.ipynb)
- Theory: [dl_fundamentals course](../../dl_fundamentals/index.html) (chapter 10, sequences) ·
  [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)